# Assignment 11 – Boosting, PCA, UMAP and KNN

---
## Question 1: Boosting — Explanation and Difference from Bagging

### What is Boosting?
Boosting is an ensemble machine learning technique that builds a strong classifier by combining many weak classifiers **sequentially**. Each new weak learner is trained to focus specifically on the mistakes made by the previous learners — misclassified examples are given higher importance (weight) so the next model pays more attention to them.

**Final prediction = weighted vote of all weak learners** — models that performed better on harder examples get a higher vote weight.

### How Boosting Works — Step by Step
1. Train a weak learner (e.g., a decision stump with depth=1) on the original data. All samples start with equal weight.
2. Identify misclassified samples. Increase their weights so the next model focuses on them.
3. Train a new weak learner on the re-weighted dataset.
4. Repeat for T rounds. Each round, another learner corrects the current ensemble's remaining errors.
5. Final prediction = weighted sum of all weak learners.

### Boosting vs Bagging — Key Differences
| Aspect | Bagging | Boosting |
|---|---|---|
| Training order | Independent, parallel | Sequential — each depends on previous errors |
| Sample weighting | Equal weights in each bootstrap | Dynamic — misclassified samples get higher weights |
| Error focus | Random subset of data | Specifically targets current ensemble's errors |
| Primary effect | Reduces **variance** (prevents overfitting) | Reduces **bias** (corrects underfitting) |
| Overfitting risk | Resistant | Can overfit with too many rounds |
| Final prediction | Equal-weight average / majority vote | Weighted combination |
| Popular algorithms | Random Forest | AdaBoost, XGBoost, LightGBM, CatBoost |


---
## Question 2: Dimensionality Reduction — Definition and Importance

### What is Dimensionality Reduction?
Dimensionality reduction transforms data from a high-dimensional space to a lower-dimensional representation that still captures the essential patterns. There are two main approaches:

- **Feature Selection:** Selecting a subset of the original features and discarding the rest. (RFE, SelectKBest, Lasso)
- **Feature Extraction / Projection:** Transforming original features into NEW lower-dimensional combinations. (PCA, UMAP, t-SNE, Autoencoders)

### Why Dimensionality Reduction is Important
- **Curse of Dimensionality:** In very high-dimensional spaces, all points appear roughly equidistant — making distance-based algorithms (KNN, SVM, clustering) unreliable.
- **Reduced computational cost:** Training time often scales quadratically or worse with features. Reducing 500 features to 50 can make training 10–100x faster.
- **Removes redundant/correlated features:** E.g., height in cm and height in inches are perfectly correlated — one adds no new information.
- **Improved model generalization:** Fewer irrelevant/noisy features reduce overfitting and improve test accuracy.
- **Visualization:** PCA, UMAP, t-SNE project high-dimensional data to 2D for visual exploration of clusters and outliers.
- **Noise removal:** Low-variance dimensions often capture noise rather than signal. Discarding them improves model performance.


---
## Question 3: Variance in PCA and Why We Maximize It

### What is Variance in PCA?
In PCA, variance refers to how spread out the data points are along a particular direction (axis). A direction with high variance captures significant variation between data points.

PCA finds new axes called **Principal Components**:
- **PC1** = direction along which the data has the **maximum variance**.
- **PC2** = direction with the second highest variance, constrained to be **orthogonal** to PC1.

`PC1 = direction w such that Var(w^T X) is maximized, subject to ||w|| = 1`

### Why We Maximize Variance in PCA
- **Variance = Information:** Directions of high variance clearly separate data points and are more informative. Zero-variance directions tell us nothing.
- **Preserving structure:** Projecting onto directions of maximum variance ensures we retain clusters, outliers, and overall data structure.
- **Signal vs Noise:** Meaningful patterns (signal) span high-variance directions. Noise contributes small, random variance. PCA naturally filters out noise.
- **Optimal reconstruction:** Principal components minimize mean squared reconstruction error — equivalently, they maximize variance. Both objectives are the same.
- **Mathematical foundation:** PCs are the **eigenvectors** of the covariance matrix `C = (1/n) X^T X`. Eigenvalues represent variance explained by each component.

`Explained Variance Ratio = λᵢ / (λ₁ + λ₂ + ... + λₙ)`


---
## Question 4: Why UMAP Performs Well for Visualization

### What is UMAP?
UMAP (Uniform Manifold Approximation and Projection) is a non-linear dimensionality reduction algorithm designed for visualization. It reduces high-dimensional data to 2D or 3D while preserving both **local structure** (nearby points stay close) and **global structure** (overall topology of the data manifold).

### Why UMAP Performs Well
- **Preserves local AND global structure:** Unlike t-SNE (only local), UMAP also preserves relative positions of distant clusters — you can meaningfully compare inter-cluster distances in a UMAP plot.
- **Mathematically grounded:** Based on Riemannian geometry and algebraic topology. Models data as lying on a low-dimensional manifold embedded in high-dimensional space — giving more trustworthy embeddings.
- **Scalability and speed:** UMAP is ~O(n log n) — handles millions of points. t-SNE is O(n²) and impractical beyond ~50,000 points.
- **Fuzzy set construction:** UMAP builds a weighted graph where nearby points have high connection weights, then optimizes a 2D embedding matching this graph (using cross-entropy loss).
- **Better cluster separation:** UMAP produces tighter, more separated clusters than t-SNE, making distinct groups easier to identify visually.
- **Hyperparameter robustness:** Less sensitive to `n_neighbors` and `min_dist` than t-SNE is to its `perplexity` parameter.
- **Preserves density:** `min_dist` controls how tightly points are packed — lower values reveal fine-grained structure; higher values show global topology.


---
## Question 5: Why KNN is Called a Lazy Learner

KNN (K-Nearest Neighbors) is called a **Lazy Learner** (or instance-based learner) because it performs **no explicit learning or model building** during the training phase — it simply stores the entire training dataset in memory and defers all computation to prediction time.

### Contrast with Eager Learners
- **Eager learners (SVM, Logistic Regression, Neural Networks):** Learn a compact model (weights, support vectors, coefficients) during training. Raw training data can be discarded after training.
- **KNN (Lazy):** During 'training', KNN does NOTHING except store all training examples. All work happens at prediction time.

### What Happens at Prediction Time
For each new test point, KNN searches the **entire training dataset** to find the K closest training points using a distance metric (usually Euclidean distance), then:
- **Classification:** Assigns the majority class among K neighbors.
- **Regression:** Assigns the mean (or weighted mean) of K neighbors' target values.

### Consequences of Being Lazy
| Aspect | Detail |
|---|---|
| Training time | O(1) — no computation during training |
| Prediction time | O(n × d) — must scan all n training points across d dimensions |
| Memory | O(n × d) — entire training dataset must stay in RAM |
| Model | No parameters, no feature importances, no decision rules (black box) |
| Adaptability | Automatically adapts to any pattern — universal approximator |


---
## Question 6: Why Feature Scaling is Important in KNN

Feature scaling is **critical** in KNN because KNN relies entirely on distance metrics. If features are on different scales, larger-scaled features will **dominate** the distance calculation, causing the algorithm to effectively ignore smaller-scaled features.

### Concrete Example — Without Scaling
Two features for predicting house prices:
- Feature 1: Area (sq ft) — range: 500–5000
- Feature 2: Number of rooms — range: 1–10

Distance between A=(2000, 3) and B=(2050, 8):
```
Distance = sqrt((2000-2050)² + (3-8)²) = sqrt(2500 + 25) ≈ 50.25
```
Area difference contributes **99%** of the distance! Rooms are effectively ignored.

### With Scaling
After StandardScaler, both features have mean=0 and std=1 — each contributes proportionally based on statistical significance, not raw magnitude.

### Common Scaling Methods for KNN
| Method | Formula | Best for |
|---|---|---|
| StandardScaler (z-score) | (x - mean) / std | Normally distributed data |
| MinMaxScaler | (x - min) / (max - min) → [0,1] | Bounded values; sensitive to outliers |
| RobustScaler | (x - median) / IQR | Datasets with significant outliers |

> **Rule:** Always scale features before applying KNN. Failing to scale is one of the most common and impactful mistakes in KNN applications.


---
## Question 7: PCA on Breast Cancer Dataset

**Analysis — Results:**
- **Components for 95% variance:** ~10 principal components explain 95% of the total variance in the breast cancer dataset (out of 30 original features). PCA reduces dimensionality by 67% while retaining 95% of information.
- **2-component visualization:** PC1 ≈ 44%, PC2 ≈ 19% of variance. Despite only 44% total variance, the 2D scatter plot shows excellent visual separation between malignant and benign tumors.
- **Features contributing most to PC1:** Size-related features dominate: mean radius, mean perimeter, mean area, mean concave points, and worst radius. These are all correlated measurements of tumor size and shape — PCA correctly identifies them as the primary axis of variation.


In [ ]:
# Question 7 – PCA on Breast Cancer Dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Load dataset
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
print(f'Original shape: {X.shape}')  # (569, 30)

# Standardize features (PCA is sensitive to scale)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA — all 30 components first
pca_full = PCA(n_components=30)
pca_full.fit(X_scaled)

# How many components explain 95% variance?
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_95 = np.argmax(cumvar >= 0.95) + 1
print(f'Components for 95% variance: {n_95}')

# Plot 1: Explained variance per component
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.bar(range(1, 31), pca_full.explained_variance_ratio_ * 100, color='steelblue')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance (%)')
plt.title('Explained Variance per Component')
plt.xticks(range(1, 31, 2))

# Plot 2: Cumulative variance
plt.subplot(1, 2, 2)
plt.plot(range(1, 31), cumvar * 100, 'bo-', lw=2)
plt.axhline(y=95, color='red', linestyle='--', label='95% threshold')
plt.axvline(x=n_95, color='green', linestyle='--', label=f'n={n_95} components')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Variance (%)')
plt.title('Cumulative Explained Variance')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Reduce to 2 components for visualization
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_scaled)
print(f'Variance explained by 2 PCs: {sum(pca_2d.explained_variance_ratio_)*100:.1f}%')

# Visualize 2D projection
plt.figure(figsize=(8, 6))
colors = ['blue', 'red']
labels = ['Malignant', 'Benign']
for i, (color, label) in enumerate(zip(colors, labels)):
    mask = y == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=color, alpha=0.6, s=40, label=label)
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('Breast Cancer Dataset — PCA 2D Projection')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Which features contribute most to PC1?
loadings = pd.Series(abs(pca_2d.components_[0]),
                     index=cancer.feature_names).sort_values(ascending=False)
print('\nTop 5 features contributing to PC1:')
print(loadings.head())


---
## Question 8: KNN on Iris Dataset — Finding Optimal K

**Where Overfitting Occurs:**  
At **small K** (K=1, 2, 3). At K=1, KNN memorizes every training point perfectly (100% training accuracy) but the decision boundary is extremely jagged and noise-sensitive. Test accuracy at K=1 is noticeably lower than at moderate K values.  
*Sign: Training accuracy much higher than test accuracy (large generalization gap).*

**Where Underfitting Occurs:**  
At **large K** (K=15–20). When K is very large, the neighborhood includes too many points — the model averages over too broad a region, creating an overly simplistic decision boundary that misses finer class structure.  
*Sign: Both training AND test accuracy decline and converge to a low value.*

**Optimal K for Iris** is typically **K=5 to 7**, where test accuracy is maximized and the bias-variance trade-off is best balanced.


In [ ]:
# Question 8 – KNN on Iris Dataset: Finding Optimal K

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Load Iris dataset
iris = load_iris()
X, y = iris.data, iris.target

# Scale features (critical for KNN)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Train KNN for K = 1 to 20
train_accs, test_accs = [], []
for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, knn.predict(X_train)))
    test_accs.append(accuracy_score(y_test,  knn.predict(X_test)))

# Find optimal K (highest test accuracy)
optimal_k = np.argmax(test_accs) + 1
print(f'Optimal K: {optimal_k}')
print(f'Best test accuracy: {max(test_accs)*100:.2f}%')

# Plot
plt.figure(figsize=(10, 6))
plt.plot(range(1, 21), [a*100 for a in train_accs], 'b-o', label='Train Accuracy')
plt.plot(range(1, 21), [a*100 for a in test_accs],  'r-o', label='Test Accuracy')
plt.axvline(x=optimal_k, color='green', linestyle='--',
            label=f'Optimal K={optimal_k}')
plt.xlabel('K (Number of Neighbors)'); plt.ylabel('Accuracy (%)')
plt.title('KNN: Train vs Test Accuracy for K=1 to 20')
plt.legend(); plt.grid(True, alpha=0.3)
plt.xticks(range(1, 21))
plt.tight_layout(); plt.show()


---
## Question 9: Fintech Company — 500 Features, 2M Rows, Severe Class Imbalance

### 1. Would You Use PCA?
**YES** — PCA is strongly recommended.
- 500 features is extremely high-dimensional with likely many correlated financial ratios. PCA removes this redundancy.
- Reducing to 50–100 components (95% variance) can make training 5–10x faster with minimal accuracy loss.
- PCA removes noise — low-variance features in financial data often represent measurement noise.
- **Caveat:** Scale all features with `StandardScaler` BEFORE PCA. Use `PCA(n_components=0.95)` to auto-select components.
- **PCA does NOT help with class imbalance** — still need SMOTE, `class_weight='balanced'`, or resampling.

### 2. Would KNN Be a Good Choice?
**NO** — KNN is a **poor choice** for this dataset:
- **Scale:** Computing distance to all 2M training points per prediction is impractical for real-time fintech decisions (payments, fraud alerts need millisecond response).
- **Memory:** Entire 2M × 500 matrix must stay in RAM (~8 GB for float64).
- **Class imbalance:** Almost every neighborhood will be dominated by legitimate transactions — KNN will nearly always predict legitimate, missing all fraud.
- **Curse of dimensionality:** Distances become less meaningful in high dimensions, further degrading KNN performance.

### 3. Which Boosting Model?
**LightGBM** (preferred for this scale):
- Histogram-based splitting and leaf-wise tree growth makes it 10–20x faster than standard GBM on large datasets.
- Built-in class imbalance handling: `is_unbalance` or `class_weight` parameters automatically up-weight the minority (fraud) class.
- Less memory than XGBoost through histogram approximation — critical at 2M rows.

### 4. How to Reduce Computation Cost?
- **PCA first:** Reduce 500 → 50–100 features.
- **Use LightGBM:** Histogram-based algorithm avoids repeated data sorting. Enable `num_threads` for multi-core.
- **Subsampling (`bagging_fraction`):** Train on 80% of rows per iteration.
- **Feature subsampling (`feature_fraction`):** Use random feature subset per tree — reduces computation and overfitting.
- **Early stopping (`early_stopping_rounds=50`):** Stop when validation loss stops improving.
- **Distributed training:** Dask-ML or Spark MLlib for distributing training across multiple machines.


---
## Question 10: Cancer Classifier — Explainability, 30 Features

### 1. Would PCA Be Useful?
**Conditionally useful — with an important caveat about explainability.**

- **Benefits:** Removes correlated features (radius, perimeter, area are all size-correlated). Improves generalization by removing noise. Slight training speedup.
- **Critical limitation:** PCA **destroys explainability**. The original 30 features (radius, texture, perimeter) are transformed into abstract principal components (PC1, PC2) with no direct medical interpretation. You can no longer say 'the model predicted malignant because tumor radius was large.'
- **Recommendation:** Avoid PCA for the final model when explainability is required. Use it only for EDA/visualization. Instead, use **feature selection** methods (RFE, SHAP-based selection) that keep original interpretable features.

### 2. Would Boosting Be Appropriate?
**YES — with SHAP explanations for explainability.**
- **Accuracy:** Boosting (XGBoost, LightGBM) typically achieves the highest accuracy on tabular datasets. Critical for medical classification.
- **Feature importance:** Boosting provides feature importance scores showing which features (e.g., worst concave points, worst perimeter) drive predictions.
- **SHAP values:** Tools like SHAP (SHapley Additive exPlanations) generate per-prediction explanations: *'This tumor was classified as malignant because worst_radius=18.5 contributed +0.4 to prediction.'* This provides medical-grade explainability.
- **Recommendation:** Use **XGBoost + SHAP** — best-in-class accuracy AND interpretable per-prediction explanations that clinicians can review and trust.

### 3. Would KNN Be Reliable?
**Partially reliable — adequate but with notable concerns.**
- **Accuracy:** KNN typically achieves 95–97% accuracy on the breast cancer dataset — competitive on this clean, well-structured data.
- **Explainability:** KNN is intuitively explainable: *'This tumor was classified as malignant because its 5 nearest neighbors in training were all malignant tumors with similar radius and texture.'* — a form of case-based reasoning clinicians may find intuitive.
- **Concerns for production:** (a) Requires storing and searching all training cases at every inference; (b) Adding new cases slows future predictions; (c) Performance degrades with correlated/irrelevant features; (d) No model parameters or feature weights for systematic explainability.
- **Recommendation:** KNN is acceptable for a prototype/research tool. For production clinical decision support, prefer **SHAP-enhanced XGBoost** or **Logistic Regression with L1 regularization** (sparse, interpretable coefficients).
